## Merging data from gamelist.xml files

in hopes that there is some missing stuff

- gamelist.xml files are in lists/<platform_name>/gamelist.xml
- platform name mappings in lists/systems.
- determine gamelist.xml tags to determine mapping
- prepare data from gamelist.xml files (to csv?)
- merge data to main

In [1]:
import pandas as pd
import numpy as np
from utils.gamelist_parser import load_all_gamelists
from utils.merge_pipeline import run_merge_pipeline, SourceConfig, CANONICAL_SCHEMA, DEFAULT_COLUMNS, KEY_COLUMNS
from utils.resolvers import resolver as merge_resolver

### Step 1: Parse all gamelist.xml files

In [2]:
# Load and parse all gamelist.xml files
gamelist_df = load_all_gamelists(lists_dir="lists", systems_file="lists/systems.txt")
gamelist_df.head()

Loaded 217 games from n64 (Nintendo 64)
Loaded 7 games from zxspectrum (Sinclair ZX Spectrum)
Loaded 10 games from dreamcast (Sega Dreamcast)
Loaded 265 games from psx (Sony PlayStation)
Loaded 564 games from msx (MSX)
Loaded 0 games from xbox (Microsoft Xbox)
Loaded 178 games from pc (IBM PC)
Loaded 29 games from c64 (Commodore 64)
Loaded 67 games from ps2 (Sony PlayStation 2)
Loaded 548 games from gbc (Nintendo Game Boy Color)
Loaded 1204 games from gba (Nintendo Game Boy Advance)
Loaded 574 games from tg16 (NEC TurboGrafx-16)
Loaded 6 games from doom (Doom)
Loaded 189 games from neogeo (SNK Neo Geo)
Loaded 263 games from gamegear (Sega Game Gear)
Loaded 75 games from nds (Nintendo DS)
Loaded 976 games from snes (Nintendo SNES (Super Nintendo))
Loaded 842 games from fba (FinalBurn Alpha)
Loaded 290 games from pcengine (NEC PC Engine)
Loaded 20 games from scummvm (ScummVM Game Engine)
Loaded 1928 games from mame (Multiple Arcade Machine Emulator)
Loaded 1024 games from genesis (Sega G

,platform,name,filename,summary,release_date,developer,publisher,genres,players,user_rating
0,Nintendo 64,007 : The World is Not Enough,007 - The World Is Not Enough,The World Is Not Enough is a first-person shoo...,20001101T000000,Eurocom Developments,Electronic Arts,Shooter / 1St Person,4.0,6.5
1,Nintendo 64,1080 TenEighty Snowboarding,1080 Snowboarding (JU) [!],1080Â° Snowboarding is a snowboard racing game...,19980401T000000,Nintendo,Nintendo,Sports / Skiing,2.0,8.5
2,Nintendo 64,40 Winks,40 Winks (Proto),There's only 40 Winks left in the world of dre...,19991114T000000,Eurocom,Piko Interactive,Action,2.0,8.0
3,Nintendo 64,A Bug's Life,"A, Bug's Life","Based on the animated film, A Bug's Life is an...",19990430T000000,Travellersâ Tales,Activision,Platform,NaN,6.0
4,Nintendo 64,Aerofighter Assault,AeroFighters Assault,The AeroFighters Assault team needs your help ...,19971030T000000,Paradigm Entertainment,Video System,Shooter / Plane,2.0,7.0


In [3]:
# Check data quality
print(f"Total games: {len(gamelist_df)}")
print(f"\nPlatforms ({gamelist_df['platform'].nunique()}):")
print(gamelist_df['platform'].value_counts().head(10))
print(f"\nNull counts:")
print(gamelist_df.isnull().sum())
print(f"\nData types:")
print(gamelist_df.dtypes)

Total games: 16258

Platforms (37):
platform
Nintendo Family Computer            1948
Multiple Arcade Machine Emulator    1928
Super Famicom                       1472
Nintendo Game Boy Advance           1204
Nintendo Entertainment System       1184
Sega Genesis                        1024
Nintendo SNES (Super Nintendo)       976
Sega Mega Drive                      954
FinalBurn Alpha                      842
NEC TurboGrafx-16                    574
Name: count, dtype: int64

Null counts:
platform           0
name               0
filename           0
summary          528
release_date    1389
developer       1243
publisher        615
genres          1059
players         2509
user_rating     1475
dtype: int64

Data types:
platform         object
name             object
filename         object
summary          object
release_date     object
developer        object
publisher        object
genres           object
players         float64
user_rating     float64
dtype: object


### Step 2 gamelist source config

In [4]:
gamelist_config = SourceConfig(
    name="gamelist",
    loader=lambda: gamelist_df,  # Use the already-loaded dataframe
    dropna_subset=["name", "platform"],
    dedupe_sort=["user_rating"],  # Prefer entries with ratings
    dedupe_subset=["name", "platform"],
)

### Step 3: Integrate with main

In [5]:
# Option B: Load existing merged data and add gamelist source
import pickle

with open('merged_df.pkl', 'rb') as f:
    merged_df = pickle.load(f)

print(f"Existing merged data: {len(merged_df)} games")
print(f"Gamelist data: {len(gamelist_df)} games")

Existing merged data: 248969 games
Gamelist data: 16258 games


In [6]:
# Merge into existing data
final_merged = run_merge_pipeline(
    main_config=SourceConfig(
        name="merged_base",
        loader=lambda: merged_df,
    ),
    source_configs=[gamelist_config],
    target_columns=DEFAULT_COLUMNS,
    key_columns=KEY_COLUMNS,
    resolver_map=merge_resolver,
    schema=CANONICAL_SCHEMA,
)

print(f"\nFinal merged data: {len(final_merged)} games")
print(f"Added {len(final_merged) - len(merged_df)} new games")

/home/jack/Projects/video-game-metadata/utils/merge_pipeline.py:198: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  stacked = pd.concat([main_df, source_df], ignore_index=True, sort=False)



Final merged data: 260800 games
Added 11831 new games


In [8]:
final_merged.head()

,name,platform,filename,summary,release_date,release_year,genres,developer,publisher,players,cooperative,rating,user_rating
0,! That Bastard Is Trying To Steal Our Gold !,Windows,NaN,Steal gold from the Lerpikon's dungeons! Get r...,NaT,<NA>,"Platform, Puzzle",WTFOMGames,WTFOMGames,<NA>,False,NaN,8.000000
1,!!!Ants!!!,Tandy TRS-80,NaN,In Ants you can become an ant and join the ran...,NaT,1979,NaN,Brian Rotolante,Synergistic Solar Inc.,1,False,NaN,5.000000
2,"""300""",Pinball,NaN,"""300"" (the exact machine name includes the quo...",NaT,1975,Pinball,Gottlieb,Gottlieb,4,False,NaN,6.944444
3,"""8 Ball""",Pinball,NaN,A pool table themed pinball game from Williams...,NaT,1952,Pinball,Williams Electronic Games,Williams Electronic Games,2,False,NaN,7.250000
4,"""History in the Making"": The First Three Years",Amstrad CPC,NaN,A celebration of U.S. Gold's first three years...,NaT,1988,Compilation,NaN,U.S. Gold,4,False,NaN,6.972222


## Step 5: Save updated merged data

In [7]:
# Save the updated merged data
with open('merged_df_with_gamelist.pkl', 'wb') as f:
    pickle.dump(final_merged, f)

print("Saved merged data to merged_df_with_gamelist.pkl")

Saved merged data to merged_df_with_gamelist.pkl
